In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

import shap

import importlib.util
import sys

# Force python to find and load the exact file on your Desktop
spec = importlib.util.spec_from_file_location(
    "data_loader",
    "/Users/new/Desktop/insurance-risk-analytics/src/data_loader.py",
)
data_loader = importlib.util.module_from_spec(spec)
sys.modules["data_loader"] = data_loader
spec.loader.exec_module(data_loader)

# Assign the function manually
load_data = data_loader.load_data


In [4]:
df = load_data('../data/MachineLearningRating_v3.txt')

/Users/new/Desktop/insurance-risk-analytics/src/data_loader.py:8: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=separator)


Data loaded successfully.
Shape: (1000098, 52)


In [5]:
df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [6]:
claims_df = df[df['TotalClaims'] > 0].copy()

In [7]:
claims_df['VehicleAge'] = 2015 - claims_df['RegistrationYear']

In [8]:
claims_df['Margin'] = (
    claims_df['TotalPremium']
    - claims_df['TotalClaims']
)

In [9]:
features = [
    'Province',
    'VehicleType',
    'make',
    'Gender',
    'VehicleAge',
    'CustomValueEstimate',
    'CalculatedPremiumPerTerm'
]

In [10]:
X = claims_df[features]

y = claims_df['TotalClaims']

In [11]:
categorical_cols = [
    'Province',
    'VehicleType',
    'make',
    'Gender'
]

numerical_cols = [
    'VehicleAge',
    'CustomValueEstimate',
    'CalculatedPremiumPerTerm'
]

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            SimpleImputer(strategy='median'),
            numerical_cols
        ),
        
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_cols
        )
    ]
)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
lr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)

In [15]:
lr_rmse = np.sqrt(
    mean_squared_error(y_test, lr_preds)
)

lr_r2 = r2_score(y_test, lr_preds)

print("Linear Regression RMSE:", lr_rmse)
print("Linear Regression R²:", lr_r2)

Linear Regression RMSE: 39124.411832241174
Linear Regression R²: 0.04820673708692935


In [16]:
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    
    ('model', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

In [17]:
rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_preds)
)

rf_r2 = r2_score(y_test, rf_preds)

print("Random Forest RMSE:", rf_rmse)
print("Random Forest R²:", rf_r2)

Random Forest RMSE: 36691.75270958301
Random Forest R²: 0.16288736193807085


In [19]:
xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    
    ('model', XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

In [20]:
xgb_rmse = np.sqrt(
    mean_squared_error(y_test, xgb_preds)
)

xgb_r2 = r2_score(y_test, xgb_preds)

print("XGBoost RMSE:", xgb_rmse)
print("XGBoost R²:", xgb_r2)

XGBoost RMSE: 36219.62878982508
XGBoost R²: 0.18429152812358696


In [21]:
results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'XGBoost'
    ],
    
    'RMSE': [
        lr_rmse,
        rf_rmse,
        xgb_rmse
    ],
    
    'R2': [
        lr_r2,
        rf_r2,
        xgb_r2
    ]
})

results.sort_values(by='R2', ascending=False)

,Model,RMSE,R2
2,XGBoost,36219.628790,0.184292
1,Random Forest,36691.752710,0.162887
0,Linear Regression,39124.411832,0.048207


In [23]:
X_train_processed = preprocessor.fit_transform(X_train)

feature_names = (
    numerical_cols +
    list(
        preprocessor.named_transformers_['cat']
        .named_steps['encoder']
        .get_feature_names_out(categorical_cols)
    )
)

X_train_processed = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, 'toarray')
    else X_train_processed,
    
    columns=feature_names
)